# 5 · Global map

Display the interactive map. For each country, period and percentile it shows either the IQB score, when that score is within ±0.05 of the true value with 95% confidence, or the 95% range of plausible scores.

In [ ]:
import json
import math

import numpy as np
import pandas as pd
import requests

import iqb_tools as T

TAB = T.TAB  # tables go to outputs/tables
SITE = T.ROOT / "map"  # the web page and its region files, published with GitHub Pages
SITE.mkdir(exist_ok=True)

LEVELS = (25, 50, 75, 95)
cfg = T.config_table()
tests = T.distinct_tests(cfg)
ctx = T.read_table("country_context").set_index("country_code")  # written by 0-setup
plan = T.read_table("pooling_plan").set_index("country_code")  # written by 1a: typical pooling per country

# 2025 pass rates per country: the test-weighted average of the 12 months
months = pd.date_range("2025-01-01", "2025-12-01", freq="MS")
df25 = T.load_level("country", months=months)
pr = T.pass_rates(df25, tests)
C = T.pool_pass_rates(df25, pr, "country_code", tests)
C["country"] = ctx.country.reindex(C.index)
pr[["cc", "month", "n_down", "n_up"]] = df25[["country_code", "month", "n_down", "n_up"]]
print(f"{len(C)} countries; standard ±{T.TOL} at {T.CONF:.0%}, design effect {T.DEFF}, minimum {T.MIN_TESTS:,} tests")

## 1. Full year 2025

In [ ]:
rows = []
for cc in C.index:
    row = C.loc[cc]
    nb = int(min(row.n_down, row.n_up))
    counts = {"tests": nb, "n_down": int(row.n_down), "n_up": int(row.n_up)}
    if nb < T.MIN_TESTS:
        rows.append({"country_code": cc, "period": "2025", "status": "pool" if nb >= T.FLOOR_TESTS else "none", **counts})
        continue
    u = T.score_uncertainty(row, row.n_down, LEVELS, key=f"{cc}|2025")
    for lv, x in u.iterrows():
        rows.append({"country_code": cc, "period": "2025", "level": lv, "status": "score" if x.conf >= T.CONF else "range",
                     "score": x.score, "conf": x.conf, "lo": x.lo, "hi": x.hi, "closest_test": x.closest_test,
                     "closest_pass_rate": x.closest_pass_rate, **counts, "window": "Jan 2025–Dec 2025"})
annual = pd.DataFrame(rows)

# Check: every P50 decision follows the publication rules (iqb_tools.display_decision, as in 1b)
RULES = {"show the score": "score", "show a range": "range", "pool more data": "pool", "no score (too few tests)": "none"}
a50 = annual[(annual.level == 50) | annual.level.isna()].set_index("country_code")
expected = [RULES[T.display_decision(r.tests, r.conf if r.level == 50 else 0)] for r in a50.itertuples()]
mismatch = (a50.status != expected).sum()
print(f"P50 decisions not following the rules: {mismatch}; {a50.status.value_counts().to_dict()}")
assert mismatch == 0

## 2. December 2025 (pooled where needed)

In [ ]:
# December 2025: the shortest trailing window ending in December with the minimum number of tests
# in both directions, at most 12 months (the pooling rule of 1a)
nd = df25.pivot(index="country_code", columns="month", values="n_down").reindex(columns=months).fillna(0)
nu = df25.pivot(index="country_code", columns="month", values="n_up").reindex(columns=months).fillna(0)
ok = (nd.iloc[:, ::-1].cumsum(axis=1) >= T.MIN_TESTS) & (nu.iloc[:, ::-1].cumsum(axis=1) >= T.MIN_TESTS)
dec_k = pd.Series(np.where(ok.any(axis=1), ok.to_numpy().argmax(axis=1) + 1, np.nan), index=nd.index)

rows, dec_pooled = [], {}
for cc in C.index:
    k = dec_k.get(cc, np.nan)
    if not np.isfinite(k):
        rows.append({"country_code": cc, "period": "Dec 2025", "status": "none",
                     "n_down": int(C.loc[cc, "n_down"]), "n_up": int(C.loc[cc, "n_up"])})
        continue
    span = months[len(months) - int(k):]
    window = f"{span[0]:%b %Y}–{span[-1]:%b %Y}" if k > 1 else f"{span[0]:%b %Y}"
    sel = (pr.cc == cc) & pr.month.isin(span)
    pooled = T.pool_pass_rates(pr[sel].assign(country_code=cc), pr[sel], "country_code", tests).iloc[0]
    dec_pooled[cc] = pooled
    u = T.score_uncertainty(pooled, pooled.n_down, LEVELS, key=f"{cc}|dec2025")
    for lv, x in u.iterrows():
        rows.append({"country_code": cc, "period": "Dec 2025", "level": lv,
                     "status": "score" if x.conf >= T.CONF else "range", "score": x.score, "conf": x.conf,
                     "lo": x.lo, "hi": x.hi, "closest_test": x.closest_test, "closest_pass_rate": x.closest_pass_rate,
                     "tests": int(min(pooled.n_down, pooled.n_up)), "n_down": int(pooled.n_down), "n_up": int(pooled.n_up),
                     "window": window, "months": int(k)})
december = pd.DataFrame(rows)

# Check: every December P50 decision follows the publication rules
RULES = {"show the score": "score", "show a range": "range", "pool more data": "pool", "no score (too few tests)": "none"}
d50 = december[(december.level == 50) | december.level.isna()].set_index("country_code")
expected = ["none" if r.status == "none" else RULES[T.display_decision(r.tests, r.conf)] for r in d50.itertuples()]
mismatch = (d50.status != expected).sum()
print(f"December P50 decisions not following the rules: {mismatch}; {d50.status.value_counts().to_dict()}")
assert mismatch == 0

## 3. How displayability changes with the percentile

In [ ]:
scores = pd.concat([annual, december], ignore_index=True)
scores["income_group"] = scores.country_code.map(ctx.income_group)

def tally(d, period):
    x = d[d.period == period]
    base = x[x.level.isna()]
    out = {}
    for lv in LEVELS:
        y = x[x.level == lv]
        out[f"P{lv}"] = {"score": (y.status == "score").sum(), "range": (y.status == "range").sum(),
                         "no score": len(base)}
    return pd.DataFrame(out)

pd.concat({"Full year 2025": tally(scores, "2025"), "December 2025": tally(scores, "Dec 2025")})

In [ ]:
x = scores[(scores.period == "2025") & scores.level.notna() & scores.income_group.notna()]
share_range = x.assign(range=x.status == "range").pivot_table(index="income_group", columns="level", values="range",
                                                              aggfunc="mean").reindex(T.INCOME_ORDER)
share_range.columns = [f"P{int(c)}" for c in share_range.columns]
print("Share of data-sufficient countries whose 2025 score must be shown as a range:")
share_range.rename(index=T.INCOME_SHORT).map("{:.0%}".format)

The tables show three things. For the full year, 12–20 data-sufficient countries need a range at every percentile; annual volumes resolve almost everything else.

For one month, ranges are far more common, and most common at P50 (60 countries in December 2025, against 23–31 at the other percentiles). At P50 a requirement is decided where half the tests pass, which is where a pass rate is noisiest. At P95 it is decided at 5%, which needs far fewer tests to resolve.

Who needs a range also moves with the percentile, following where each group's pass rates sit. At P25, where a requirement is decided at 75% of tests, it is mostly high- and upper-middle-income countries. At P75, decided at 25%, it is mostly low-income countries.

In [ ]:
scores.drop(columns="income_group").to_csv(TAB / "map_scores.csv", index=False)
print("wrote map_scores.csv", scores.shape)

## 4. Build the map page

We draw country shapes from Natural Earth 1:50m boundaries (public domain), projected with the Equal Earth projection and simplified to the map's resolution. Territories that M-Lab reports separately from their parent country (French Guiana, Guadeloupe, Martinique, Réunion, Mayotte, the Caribbean Netherlands, Svalbard) are split out of France, the Netherlands and Norway, and places too small to hover get a dot marker.

In [ ]:
NE = T.EXT / "naturalearth"; NE.mkdir(parents=True, exist_ok=True)
ne_path = NE / "ne_50m_admin_0_countries.geojson"
if not ne_path.exists():
    url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson"
    ne_path.write_bytes(requests.get(url, timeout=300).content)
geo = json.loads(ne_path.read_text())

A1, A2, A3, A4 = 1.340264, -0.081106, 0.000893, 0.003796

def equal_earth(lon, lat):
    lam, phi = np.radians(lon), np.radians(lat)
    th = np.arcsin(math.sqrt(3) / 2 * np.sin(phi))
    th2, th6 = th**2, th**6
    x = 2 * math.sqrt(3) * lam * np.cos(th) / (3 * (A1 + 3 * A2 * th2 + th6 * (7 * A3 + 9 * A4 * th2)))
    return x, th * (A1 + A2 * th2 + th6 * (A3 + A4 * th2))

SPLIT = {  # (code, lon/lat box) of overseas parts reported separately by M-Lab
    "FR": [("GF", (-55.5, -51.0, 1.5, 6.5)), ("GP", (-62.0, -60.9, 15.8, 16.6)), ("MQ", (-61.3, -60.7, 14.3, 14.95)),
           ("RE", (55.0, 56.0, -21.5, -20.8)), ("YT", (44.9, 45.4, -13.1, -12.5))],
    "NL": [("BQ", (-69.0, -62.5, 11.9, 17.8))],
    "NO": [("SJ", (-10.0, 35.0, 74.0, 81.5))],
}
features = {}
for f in geo["features"]:
    code, name = f["properties"]["ISO_A2_EH"], f["properties"]["NAME"]
    if name == "Antarctica" or (code == "-99" and name not in ("Somaliland", "N. Cyprus")):
        continue
    code = {"Somaliland": "SO", "N. Cyprus": "CY"}.get(name, code)  # drawn with the country M-Lab reports them under
    g = f["geometry"]
    for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
        ring0 = np.asarray(poly[0])
        cx, cy = ring0[:, 0].mean(), ring0[:, 1].mean()
        target = next((sub for sub, (x0, x1, y0, y1) in SPLIT.get(code, []) if x0 <= cx <= x1 and y0 <= cy <= y1), code)
        features.setdefault(target, []).extend(np.asarray(r) for r in poly)

proj = {c: [np.column_stack(equal_earth(r[:, 0], r[:, 1])) for r in rings] for c, rings in features.items()}
allxy = np.vstack([r for rings in proj.values() for r in rings])
xmin, ymax = allxy[:, 0].min(), allxy[:, 1].max()
scale = 1000.0 / (allxy[:, 0].max() - xmin)
height = (ymax - allxy[:, 1].min()) * scale

paths, dots = [], []
for code, rings in proj.items():
    px = [np.column_stack([(r[:, 0] - xmin) * scale, (ymax - r[:, 1]) * scale]) for r in rings]
    parts = []
    for r in px:
        if np.ptp(r[:, 0]) * np.ptp(r[:, 1]) < 0.25 and len(px) > 1:
            continue
        kept = [r[0]]
        for pt in r[1:]:
            if abs(pt[0] - kept[-1][0]) + abs(pt[1] - kept[-1][1]) >= 0.45:
                kept.append(pt)
        if len(kept) >= 3:
            parts.append("M" + "L".join(f"{a:.1f},{b:.1f}" for a, b in kept) + "Z")
    allr = np.vstack(px)
    if parts:
        paths.append((code, "".join(parts)))
    if max(np.ptp(allr[:, 0]), np.ptp(allr[:, 1])) < 6:
        big = max(px, key=lambda r: np.ptp(r[:, 0]) * np.ptp(r[:, 1]))
        dots.append((code, big[:, 0].mean(), big[:, 1].mean()))
drawn = {c for c, _ in paths} | {c for c, *_ in dots}
print(f"{len(paths)} shapes, {len(dots)} markers; viewBox 1000 × {height:.0f}")

## 5. Regions

Clicking a country on the map opens its regions. We pool M-Lab's first-level subdivision table (states, provinces, regions) over 2025 and score it with exactly the same rules as countries: the data-sufficiency minimum in both directions, then the display-confidence check at each percentile. Regions use the full year only, because a single month has too few tests in most regions.

*Caveat.* Tests are placed in regions by IP-address geolocation, which is approximate. Mobile operators often route traffic through a few central points, usually in the capital or largest city, so tests can be placed there even when the user is elsewhere (4a). The region with the most tests is often not the capital (Lagos rather than Abuja, Dar es Salaam rather than Dodoma, São Paulo rather than Brasília), and it tends to hold more tests, and possibly different users, than its population suggests. The data cannot separate mobile from fixed tests, so location error cannot be told apart from testing that really is concentrated in large cities.

In [ ]:
def r3(v):
    return None if v is None or pd.isna(v) else round(float(v), 3)

def pattern_record(row, n_tests, key):
    """What the page needs to recompute score, confidence and range for any use-case weighting:
    the ten pass rates and, per level, the observed and simulated pass/fail patterns (iqb_tools.score_patterns)."""
    pat = T.score_patterns(row, n_tests, LEVELS, key=key)
    return {"pr": [round(float(v), 4) for v in pat["pass_rates"]],
            "L": {str(lv): [x["obs"], x["dist"]] for lv, x in pat["levels"].items()}}

sub = T.load_level("country_subdivision1", months=months)
sub = sub[sub.subdivision1_iso_code.notna()]
rkeys = ["country_code", "subdivision1_iso_code"]
rnames = sub.groupby(rkeys).subdivision1_name.agg(lambda s: s.mode().iloc[0])
RG = T.pool_pass_rates(sub, T.pass_rates(sub, tests), rkeys, tests)
RG["name"] = rnames
RG["share"] = RG.n_down / RG.index.get_level_values(0).map(C.n_down)

reg_rows = []
for (cc, code), row in RG.iterrows():
    nb = int(min(row.n_down, row.n_up))
    base = {"country_code": cc, "region": code, "name": row["name"], "tests": nb,
            "n_down": int(row.n_down), "n_up": int(row.n_up), "share": row.share}
    if nb < T.MIN_TESTS:
        reg_rows.append({**base, "status": "pool" if nb >= T.FLOOR_TESTS else "none"})
        continue
    u = T.score_uncertainty(row, row.n_down, LEVELS, key=f"{cc}|{code}|2025")
    for lv, x in u.iterrows():
        reg_rows.append({**base, "level": lv, "status": "score" if x.conf >= T.CONF else "range", "score": x.score,
                         "conf": x.conf, "lo": x.lo, "hi": x.hi, "closest_test": x.closest_test,
                         "closest_pass_rate": x.closest_pass_rate})
regions = pd.DataFrame(reg_rows)
regions.to_csv(TAB / "map_region_scores.csv", index=False)

per_region = regions.drop_duplicates(["country_code", "region"])
p50r = regions[regions.level == 50]
print(f"{per_region.country_code.nunique()} countries, {len(per_region):,} regions; "
      f"{(per_region.tests >= T.MIN_TESTS).sum():,} with enough tests for a score "
      f"(P50: {(p50r.status == 'score').sum():,} shown as a score, {(p50r.status == 'range').sum():,} as a range)")
located = per_region.groupby("country_code").share.sum()
print(f"Share of a country's tests located in a region: median {located.median():.0%}")

### Region boundaries

We take region boundaries from Natural Earth's 1:10m states and provinces (public domain), as split per country in the IQB repository (`prototype/natural_earth/geojson_admin1`). M-Lab and Natural Earth do not always use the same level: the UK is reported as four nations but drawn as 232 local authorities, and France as 13 regions but drawn as 101 departments. We therefore match each country in steps:

1. **codes** (ISO 3166-2, regional and HASC codes);
2. **exact names**, then alternate names;
3. **groups** of shapes that share a parent region, merged up to M-Lab's level;
4. a **cautious fuzzy match** for spelling variants and merged regions (e.g. Mazovia/Masovian, Trøndelag = Nord- + Sør-Trøndelag).

A shape is never assigned to more than one M-Lab region. A country gets a region map when at least 70% of its tests fall in matched regions and at least half of its scored regions can be drawn; otherwise its regions are listed without a map. (The second condition matters for Kenya: Nairobi holds most tests, but its other 22 scored counties have no boundary at that level.) Alaska and Hawaii are drawn as insets.

In [ ]:
import difflib
import re

import pycountry

ADMIN1 = T.ROOT.parent / "IQB" / "iqb" / "prototype" / "natural_earth" / "geojson_admin1"
NE10 = NE / "ne_10m_admin_1_states_provinces.geojson"
if not ADMIN1.exists() and not NE10.exists():   # fall back to Natural Earth's single admin-1 file
    url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_1_states_provinces.geojson"
    NE10.write_bytes(requests.get(url, timeout=600).content)
_ne10 = None

def admin1_features(cc):
    global _ne10
    c = pycountry.countries.get(alpha_2=cc)
    if c is None:
        return None
    if ADMIN1.exists():
        path = ADMIN1 / f"{c.alpha_3}_admin1.geojson"
        return json.loads(path.read_text())["features"] if path.exists() else None
    if _ne10 is None:
        _ne10 = {}
        for f in json.loads(NE10.read_text())["features"]:
            _ne10.setdefault(f["properties"].get("iso_a2"), []).append(f)
    return _ne10.get(cc)

GENERIC = (r"\b(county|province|provincia|region|regional|governorate|state|estado|department|departamento|"
           r"voivodeship|district|prefecture|oblast|krai|kray|republic|autonomous|municipality|municipio|city|"
           r"special|capital|territory|division|emirate|parish|canton|of|the|and|national|metropolitan|federal|"
           r"island|islands|area|zone)\b")
ALIAS = {"flanders": "flemish", "wallonia": "walloon", "mazovia": "masovian"}

def fix_text(s):
    """Undo UTF-8 text that was decoded as Latin-1 ('franÃ§aise' -> 'française')."""
    if not isinstance(s, str):
        return None
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

def name_key(s):
    if not isinstance(s, str):
        return None
    base = T.norm_name(fix_text(s))
    k = re.sub(r"\s+", " ", re.sub(GENERIC, " ", base)).strip() or base
    return ALIAS.get(k, k)

def _suffix(v, sep):
    return v.split(sep, 1)[1].upper() if isinstance(v, str) and sep in v else None

def match_regions(feats, regs):
    """Owner (M-Lab region code or None) of each boundary shape."""
    P = pd.DataFrame([f["properties"] for f in feats])
    col = lambda name: P[name] if name in P else pd.Series([None] * len(P))
    codes = [col("iso_3166_2").map(lambda v: _suffix(v, "-")), col("region_cod").map(lambda v: _suffix(v, "-")),
             col("code_hasc").map(lambda v: _suffix(v, "."))]
    names = [col(n).map(name_key) for n in ("name", "gn_name", "name_en", "woe_name")]
    alt = col("name_alt").map(lambda v: [name_key(a) for a in v.split("|")] if isinstance(v, str) else [])
    groups = [col("region").map(name_key), col("geonunit").map(name_key)]
    owner, pending = [None] * len(P), []
    for code, rname in regs.items():
        ucode, key = str(code).upper(), name_key(rname)
        hit = next((np.flatnonzero((v == ucode).to_numpy()) for v in codes if (v == ucode).any()), None)
        if hit is None:
            hit = next((np.flatnonzero((v == key).to_numpy()) for v in names if (v == key).any()), None)
        if hit is None and any(key in a for a in alt):
            hit = np.flatnonzero([key in a for a in alt])
        if hit is None:
            hit = next((np.flatnonzero((v == key).to_numpy()) for v in groups if (v == key).any()), None)
        if hit is None:
            pending.append((code, key))
            continue
        for i in hit:
            owner[i] = owner[i] or code
    keys = [[k for k in (names[0][i], names[1][i], names[2][i], names[3][i], groups[0][i]) if isinstance(k, str) and k]
            for i in range(len(P))]
    claims = {}
    for code, key in pending:
        for i, ks in enumerate(keys):
            if owner[i] is None and key and len(key) >= 4 and any(
                    len(k) >= 4 and (re.search(rf"\b{re.escape(key)}\b", k) or re.search(rf"\b{re.escape(k)}\b", key)
                                     or difflib.SequenceMatcher(None, key, k).ratio() >= 0.85) for k in ks):
                claims.setdefault(i, set()).add(code)
    for i, who in claims.items():
        if len(who) == 1:            # never let one shape stand for several M-Lab regions
            owner[i] = next(iter(who))
    return owner

INSETS = {"US": {"AK": (0.35, -117.0, 27.5), "HI": (1.0, -106.0, 24.5)}}   # scale, target lon, target lat
W_PX, H_PX, GRID = 600.0, 440.0, 0.6

def region_geometry(feats, owner, insets=None):
    """Shapes per region, fitted to a 600 x 440 panel, plus one path of region borders.

    Coordinates are snapped to a 0.6-px grid, which simplifies a shared border identically
    on both sides; borders between shapes of the same region are dropped.
    """
    rings = []
    for f, o in zip(feats, owner):
        g = f["geometry"]
        if g is None:
            continue
        for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
            rings.extend([o, np.asarray(r, dtype=float)] for r in poly)
    for code, (k, tx, ty) in (insets or {}).items():
        mine = [r for o, r in rings if o == code]
        if not mine:
            continue
        for item in rings:
            if item[0] == code:
                item[1] = np.column_stack([np.where(item[1][:, 0] > 0, item[1][:, 0] - 360, item[1][:, 0]), item[1][:, 1]])
        allm = np.vstack([r for o, r in rings if o == code])
        cx, cy = allm[:, 0].mean(), allm[:, 1].mean()
        for item in rings:
            if item[0] == code:
                item[1] = np.column_stack([tx + (item[1][:, 0] - cx) * k, ty + (item[1][:, 1] - cy) * k])
    if not rings:
        return None
    lon = np.concatenate([r[:, 0] for _, r in rings])
    if lon.max() - lon.min() > 180:  # crosses the antimeridian
        rings = [[o, np.column_stack([np.where(r[:, 0] < 0, r[:, 0] + 360, r[:, 0]), r[:, 1]])] for o, r in rings]
    core = np.vstack([r for o, r in rings if o is not None] or [r for _, r in rings])
    x0, x1, y0, y1 = core[:, 0].min(), core[:, 0].max(), core[:, 1].min(), core[:, 1].max()
    mx, my = (x1 - x0) * 0.15 + 0.5, (y1 - y0) * 0.15 + 0.5
    rings = [(o, r) for o, r in rings if o is not None or (r[:, 0].max() >= x0 - mx and r[:, 0].min() <= x1 + mx
                                                           and r[:, 1].max() >= y0 - my and r[:, 1].min() <= y1 + my)]
    allr = np.vstack([r for _, r in rings])
    lat0 = math.radians((allr[:, 1].min() + allr[:, 1].max()) / 2)
    lonmin, latmax = allr[:, 0].min(), allr[:, 1].max()
    xs, ys = (allr[:, 0] - lonmin) * math.cos(lat0), latmax - allr[:, 1]
    s = min(W_PX / max(xs.max(), 1e-9), H_PX / max(ys.max(), 1e-9))

    snapped, edge_regions, edge_count = [], {}, {}
    for o, r in rings:
        q = np.round(np.column_stack([(r[:, 0] - lonmin) * math.cos(lat0) * s, (latmax - r[:, 1]) * s]) / GRID).astype(np.int64)
        q = q[np.r_[True, np.any(q[1:] != q[:-1], axis=1)]]
        if len(q) < 4:
            continue
        snapped.append((o, q))
        for a, b in zip(map(tuple, q[:-1]), map(tuple, q[1:])):
            e = (a, b) if a < b else (b, a)
            edge_regions.setdefault(e, set()).add(o)
            edge_count[e] = edge_count.get(e, 0) + 1
    pt = lambda p: f"{p[0] * GRID:.1f},{p[1] * GRID:.1f}"
    fills, runs = {}, []
    for o, q in snapped:
        fills.setdefault(o or "", []).append("M" + "L".join(pt(p) for p in q[:-1]) + "Z")
        run = []
        for a, b in zip(map(tuple, q[:-1]), map(tuple, q[1:])):
            e = (a, b) if a < b else (b, a)
            if o is not None and edge_count[e] >= 2 and edge_regions[e] == {o}:   # inside one region
                if len(run) > 1:
                    runs.append(run)
                run = []
            else:
                run = run or [a]
                run.append(b)
        if len(run) > 1:
            runs.append(run)
    return {"w": round(xs.max() * s, 1), "h": round(ys.max() * s, 1),
            "f": [[k, "".join(v)] for k, v in fills.items()],
            "b": "".join("M" + "L".join(pt(p) for p in run) for run in runs)}

In [ ]:
RDIR = SITE / "regions"
RDIR.mkdir(parents=True, exist_ok=True)
region_info, cover_rows = {}, []
for cc, g in regions.groupby("country_code"):
    regs = g.drop_duplicates("region").set_index("region")
    records = []
    for code, x in g.groupby("region", sort=False):
        lv = x[x.level.notna()]
        rec = {"c": code, "n": x["name"].iloc[0], "t": int(x.tests.iloc[0]),
               "td": int(x.n_down.iloc[0]), "tu": int(x.n_up.iloc[0]), "sh": r3(x.share.iloc[0]),
               "s": "ok" if len(lv) else x.status.iloc[0]}
        if len(lv):
            row = RG.loc[(cc, code)]
            rec.update(pattern_record(row, row.n_down, f"{cc}|{code}|2025"))
        records.append(rec)
    feats = admin1_features(cc)
    geo, cover, not_drawn = None, 0.0, list(regs["name"])
    if feats:
        owner = match_regions(feats, regs["name"])
        drawn_codes = {o for o in owner if o}
        cover = regs["share"][regs.index.isin(drawn_codes)].sum() / max(regs["share"].sum(), 1e-12)
        scored = [code for code, x in g.groupby("region") if x.level.notna().any()]
        drawn_scored = np.mean([code in drawn_codes for code in scored]) if scored else 1.0
        if cover >= 0.7 and drawn_scored >= 0.5:
            geo = region_geometry(feats, owner, INSETS.get(cc))
            not_drawn = sorted(regs["name"][~regs.index.isin(drawn_codes)])
    payload = {"regions": records, "geo": geo, "located": r3(regs["share"].sum()), "notDrawn": not_drawn if geo else []}
    (RDIR / f"{cc}.json").write_text(json.dumps(payload, ensure_ascii=False, separators=(",", ":")))
    region_info[cc] = {"k": len(records), "m": geo is not None}
    cover_rows.append((cc, len(records), cover, geo is not None))

cov = pd.DataFrame(cover_rows, columns=["country_code", "regions", "tests matched to a boundary", "map"])
total = sum(f.stat().st_size for f in RDIR.glob("*.json"))
print(f"{len(cov)} countries with regions: {cov['map'].sum()} with a region map, {(~cov['map']).sum()} listed only; "
      f"files total {total / 1e6:.1f} MB")
print("Listed without a map:", ", ".join(C.loc[c, "country"] for c in cov.country_code[~cov["map"]] if c in C.index))

## 6. Write the page

In [ ]:
data = {}
for cc in C.index:
    inc = ctx.income_group.get(cc)
    rec = {"n": C.loc[cc, "country"], "i": T.INCOME_SHORT.get(inc, "") if isinstance(inc, str) else ""}
    for key, period in (("a", "2025"), ("d", "Dec 2025")):
        x = scores[(scores.country_code == cc) & (scores.period == period)]
        lv = x[x.level.notna()]
        u = {"s": "ok" if len(lv) else (x.status.iloc[0] if len(x) else "none")}
        if len(x) and pd.notna(x.tests.iloc[0]):
            u["t"] = int(x.tests.iloc[0])
        if len(x) and pd.notna(x.n_down.iloc[0]):
            u["td"], u["tu"] = int(x.n_down.iloc[0]), int(x.n_up.iloc[0])
        if len(lv):
            unit = C.loc[cc] if key == "a" else dec_pooled[cc]
            u.update(pattern_record(unit, unit.n_down, f"{cc}|2025" if key == "a" else f"{cc}|dec2025"))
            u["w"] = lv.window.iloc[0]
        if key == "d" and u["s"] != "ok":   # no December score: also December's own tests, for the message
            u["md"], u["mu"] = int(nd.at[cc, months[-1]]), int(nu.at[cc, months[-1]])
        if key == "d" and cc in plan.index:
            u["plan"] = plan.loc[cc, "plan"]
        rec[key] = u
    if cc in region_info:
        rec["rg"] = region_info[cc]
    data[cc] = rec

# Check: without a December score, the counts quoted are December's own and the 12 months to December (= all of 2025)
no_dec = [cc for cc, rec in data.items() if rec["d"]["s"] != "ok"]
assert all(data[cc]["d"]["md"] <= data[cc]["d"]["td"] == int(C.loc[cc, "n_down"]) and
           data[cc]["d"]["mu"] <= data[cc]["d"]["tu"] == int(C.loc[cc, "n_up"]) for cc in no_dec)
print(f"{len(no_dec)} countries without a December score; December counts added to their message")

cfg = T.config_table()
PROFILES = {  # use-case weights; "custom" starts from the default
    "default": {"label": "Default", "w": {u: 1 for u in T.USE_CASES}},
    "nogaming": {"label": "No gaming", "w": {**{u: 1 for u in T.USE_CASES}, "gaming": 0}},
    "essential": {"label": "Essential services", "w": {**{u: 1 for u in T.USE_CASES}, "online backup": 0, "gaming": 0}},
    "basic": {"label": "Basic access", "w": {u: 1 if u in ("web browsing", "video conferencing") else 0 for u in T.USE_CASES}},
    "edu": {"label": "IQB Edu", "w": {"web browsing": 1, "video streaming": 1, "audio streaming": 0.5,
                                       "video conferencing": 1, "online backup": 0.5, "gaming": 0}},
}
assert all(set(p["w"]) == set(T.USE_CASES) for p in PROFILES.values())
order = T.test_order()
meta = {"minTests": T.MIN_TESTS, "deff": T.DEFF, "tol": T.TOL, "conf": T.CONF, "B": 2000,
        "tests": order, "useCases": list(T.USE_CASES), "profiles": PROFILES,
        "config": [[r.use_case, order.index(T.test_key(r.metric, r.threshold)), r.req_weight] for r in cfg.itertuples()],
        "notOnMap": sorted(set(data) - drawn)}

# Check: the page's calculation, done here in Python for the default profile, reproduces the published scores
def page_calc(entry, w, B=2000):
    obs_mask, dist = entry
    score = lambda m: sum(w[k] for k in range(len(w)) if m >> k & 1)
    v = score(obs_mask)
    pts = sorted((score(m), c) for m, c in dist)
    conf = sum(c for x, c in pts if abs(x - v) <= T.TOL + 1e-9) / B
    def quantile(pq):
        pos = pq * (B - 1); i = int(np.floor(pos)); fr = pos - i
        def at(j):
            cum = 0
            for x, c in pts:
                cum += c
                if j < cum:
                    return x
            return pts[-1][0]
        return at(i) + (at(min(i + 1, B - 1)) - at(i)) * fr
    return v, conf, quantile(0.025), quantile(0.975)

w_default = T.effective_test_weights(cfg).reindex(order).to_numpy()
worst, checked = 0.0, 0
for cc, rec in data.items():
    for key, period in (("a", "2025"), ("d", "Dec 2025")):
        if "L" not in rec[key]:
            continue
        ref = scores[(scores.country_code == cc) & (scores.period == period)].set_index("level")
        for lv, entry in rec[key]["L"].items():
            got = page_calc(entry, w_default)
            want = ref.loc[int(lv), ["score", "conf", "lo", "hi"]].to_numpy(float)
            worst = max(worst, float(np.max(np.abs(np.array(got) - want))))
            checked += 1
print(f"page calculation vs notebook results, default profile: {checked} country-period-levels, max difference {worst:.1e}")
assert worst < 1e-9

# Check: IQB Edu scored straight from the pass rates (iqb_tools.iqb_scores) matches the page's calculation
cfg_edu = T.with_use_case_weights(cfg, PROFILES["edu"]["w"])
w_edu = T.effective_test_weights(cfg_edu).reindex(order).fillna(0).to_numpy()
ref_edu = T.iqb_scores(C, cfg_edu, q=LEVELS)
worst, checked = 0.0, 0
for cc, rec in data.items():
    for lv, entry in rec["a"].get("L", {}).items():
        worst = max(worst, abs(page_calc(entry, w_edu)[0] - ref_edu.loc[cc, int(lv)]))
        checked += 1
print(f"page calculation vs iqb_scores, IQB Edu profile: {checked} country-levels, max difference {worst:.1e}")
assert worst < 1e-9
html = (T.ROOT / "map_template.html").read_text()
html = (html.replace("/*__DATA__*/", json.dumps(data, ensure_ascii=False, separators=(",", ":")))
            .replace("/*__META__*/", json.dumps(meta, separators=(",", ":")))
            .replace("<!--__PATHS__-->", "\n".join(f'<path data-id="{c}" d="{d}"/>' for c, d in sorted(paths)))
            .replace("<!--__DOTS__-->", "\n".join(f'<circle data-id="{c}" cx="{x:.1f}" cy="{y:.1f}" r="3.2"/>'
                                                 for c, x, y in sorted(dots)))
            .replace("__VBH__", f"{height:.0f}"))
dfx = T.read_table("design_effect_series")
visible = dfx[dfx.n_month < 10_000]
quoted = {"independent tests": f"gives {T.Z**2 * 0.25 / T.TOL**2:.0f}", "series": f"{len(visible):,} monthly series",
          "median": f"median of {visible.deff.median():.1f}", "design effect": f"{T.Z**2 * 0.25 / T.TOL**2:.0f} × {T.DEFF}",
          "minimum": f"rounded up to {T.MIN_TESTS:,}"}
missing = [k for k, s in quoted.items() if s not in html]
assert not missing, f"'How to read the map' quotes outdated numbers: {missing}"
# A complete HTML document, so the page also renders correctly when served from disk
standalone = ('<!doctype html>\n<meta charset="utf-8">\n'
              '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
              '<style>[hidden] { display: none !important; }</style>\n' + html)
out = SITE / "index.html"
out.write_text(standalone)
print(f"wrote {out} ({out.stat().st_size / 1e6:.2f} MB); not drawn: {', '.join(data[c]['n'] for c in meta['notOnMap'])}")

The page loads each country's regions on demand from `map/regions/XX.json`; everything else is embedded in the page (fonts come from Google Fonts). Because of those region files it has to be opened through a web server, not by double-clicking the file. From the repository root:

```bash
python3 -m http.server 8000
```

then open http://localhost:8000/map/. 